# 042 — Signal evolution: does each refinement actually help?

`040`/`041` rank every model's *best* signal against every other model's. This notebook asks a different question, per model instead of across models: starting from raw delta, does each successive refinement — structural (SSIM) decomposition, then sigma-normalization — actually improve detection, or is it just a more complicated way of drawing the same map? Two refinement steps exist:

- **structural**: `raw delta` → `structural delta` (isolates the SSIM structure term). Available to every model, deterministic and NLL alike.
- **sigma-normalization**: `X` → `X / sigma` (divides by the model's own predicted uncertainty). Only meaningful for NLL models — deterministic architectures have no sigma.

Same ground-truth image set as `040`/`041` (see `040_signal_evaluation.ipynb`'s title cell for the full companion-notebook table) — this notebook reruns prediction independently rather than importing `040`/`041`'s results, matching how every other notebook in this project is self-contained.


Make the project root importable so `scripts.*` resolves regardless of the notebook's working directory.


In [1]:
import sys
from pathlib import Path

project_root = Path().absolute()
if project_root.name == "notebooks":
    project_root = project_root.parent
sys.path.insert(0, str(project_root))

Imports and a GPU sanity check.


In [2]:
import numpy as np
import tensorflow as tf
from PIL import Image

from scripts.calibration import laplace_sigma_from_scale, learned_zscore, structural_zscore
from scripts.config import settings
from scripts.dataset import pad_to_multiple
from scripts.delta_analysis import compute_local_stats, compute_ssim_components
from scripts.detection import rank_signals
from scripts.trainer import load_model
from scripts.trainer_nll import load_model_nll

gpus = tf.config.list_physical_devices("GPU")
print(f"GPUs available: {gpus}")

GPUs available: [PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU')]


## 1. Ground-truth images

Same generic discovery as `040`/`041` §1. Currently `GT01`, `GT02`, `GT03`.


In [3]:
TEST_RGB_DIR = project_root / "data" / "test" / "rgb"
TEST_IR_DIR = project_root / "data" / "test" / "ir"
ANNOTATIONS_DIR = project_root / "data" / "test" / "annotations"

mask_paths = (
    sorted(ANNOTATIONS_DIR.glob("*_Map.png")) if ANNOTATIONS_DIR.exists() else []
)
eval_pairs = []
for mask_path in mask_paths:
    stem = mask_path.name.removesuffix("_Map.png")
    rgb_path, ir_path = TEST_RGB_DIR / f"{stem}.jpg", TEST_IR_DIR / f"{stem}.jpg"
    if rgb_path.exists() and ir_path.exists():
        eval_pairs.append((rgb_path, ir_path))
    else:
        print(f"[skip] {stem}: mask found but rgb/ir pair missing")

print(f"Ground-truth images: {len(eval_pairs)} | {[p.stem for p, _ in eval_pairs]}")

if len(eval_pairs) < 2:
    raise RuntimeError(
        "Need at least 2 ground-truth images for the ranking's std to mean "
        "anything. Add masks to data/test/annotations/."
    )

Ground-truth images: 3 | ['GT01', 'GT02', 'GT03']


## 2. Load every model — both families

6 deterministic (`models/deterministic/`) + 4 NLL architectures, all with the unified `laplace_nll` loss (`models/nll/`, `fixing.md` #10 — Round 2 collapsed `efficientnet_unet_nll`'s previous `gaussian_nll`/`beta_nll` split the same way) — everything `040`/`041` cover, together. Missing checkpoints are skipped with a message.


In [4]:
DET_ARCHS = [
    "unet",
    "resunet",
    "attention_unet",
    "efficientnet_unet",
    "efficientnet_unet_ft",
    #"unet_v2",
    #"unet_restormer",
]
NLL_ARCHS = ["unet_nll", "resunet_nll", "attention_unet_nll", "efficientnet_unet_nll", "efficientnet_unet_nll_ft"]
BETA = settings.NLL_BETA

det_models: dict = {}
for arch in DET_ARCHS:
    try:
        det_models[arch] = load_model(
            arch, model_dir=settings.MODELS_DIR / "deterministic"
        )
    except FileNotFoundError as exc:
        print(f"[skip] {exc}")

nll_models: dict = {}
for arch in NLL_ARCHS:
    try:
        nll_models[arch] = load_model_nll(
            arch, model_dir=settings.MODELS_DIR / "nll", loss_name="laplace_nll", beta=BETA
        )
    except FileNotFoundError as exc:
        print(f"[skip] {exc}")

print(f"Deterministic: {list(det_models)}")
print(f"NLL: {list(nll_models)}")
if not det_models and not nll_models:
    raise RuntimeError("No checkpoints found under models/.")


2026-08-26 12:51:13.501124: I metal_plugin/src/device/metal_device.cc:1154] Metal device set to: Apple M4 Max
2026-08-26 12:51:13.501142: I metal_plugin/src/device/metal_device.cc:296] systemMemory: 36.00 GB
2026-08-26 12:51:13.501146: I metal_plugin/src/device/metal_device.cc:313] maxCacheSize: 14.04 GB
2026-08-26 12:51:13.501158: I tensorflow/core/common_runtime/pluggable_device/pluggable_device_factory.cc:305] Could not identify NUMA node of platform GPU ID 0, defaulting to 0. Your kernel may not have been built with NUMA support.
2026-08-26 12:51:13.501172: I tensorflow/core/common_runtime/pluggable_device/pluggable_device_factory.cc:271] Created TensorFlow device (/job:localhost/replica:0/task:0/device:GPU:0 with 0 MB memory) -> physical PluggableDevice (device: 0, name: METAL, pci bus id: <undefined>)


Deterministic: ['unet', 'resunet', 'attention_unet', 'efficientnet_unet', 'efficientnet_unet_ft']
NLL: ['unet_nll', 'resunet_nll', 'attention_unet_nll', 'efficientnet_unet_nll', 'efficientnet_unet_nll_ft']


## 3. Build every signal, per model

Deterministic models get `raw delta`/`structural delta`; NLL models get those two plus `\|z\|`/`structural z` — the sigma-normalized counterparts. Signal names are stored per model so §4 can group them back into each model's own refinement sequence.


In [5]:
def load_pair(rgb_path: Path, ir_path: Path) -> tuple[np.ndarray, np.ndarray]:
    rgb = np.array(Image.open(rgb_path).convert("RGB")).astype(np.float32) / 255.0
    ir = np.array(Image.open(ir_path).convert("L")).astype(np.float32) / 255.0
    return rgb, ir


MASK_THRESHOLD = 127


def load_mask(stem: str) -> np.ndarray:
    mask_path = ANNOTATIONS_DIR / f"{stem}_Map.png"
    return np.array(Image.open(mask_path).convert("L")) > MASK_THRESHOLD


def structural_delta_map(ir: np.ndarray, pred: np.ndarray) -> np.ndarray:
    stats = compute_local_stats(ir, pred)
    return 1.0 - compute_ssim_components(stats).structure


def sigma_for(log_scale: np.ndarray) -> np.ndarray:
    """Convert a model's raw second (log-scale) channel to a true std dev.

    Every loaded NLL architecture trains as Laplace since ``fixing.md``
    #10's Round 2 collapse, so ``laplace_sigma_from_scale`` (``b * sqrt(2)``)
    applies uniformly — no more per-model distribution dispatch needed.
    """
    return laplace_sigma_from_scale(np.exp(log_scale))


# model name -> ordered list of (stage label, signal name)
STAGES: dict[str, list[tuple[str, str]]] = {}


def signals_for(rgb: np.ndarray, ir: np.ndarray) -> dict[str, np.ndarray]:
    padded, _ = pad_to_multiple(tf.constant(rgb), multiple=settings.PATCH_MULTIPLE)
    h, w = ir.shape
    batch = padded[tf.newaxis, ...]

    signals = {}
    for name, model in det_models.items():
        pred = model.predict(batch, verbose=0)[0, :h, :w, 0]
        raw_name, struct_name = f"{name} [raw delta]", f"{name} [structural delta]"
        signals[raw_name] = np.abs(ir - pred)
        signals[struct_name] = structural_delta_map(ir, pred)
        STAGES.setdefault(name, [("raw", raw_name), ("structural", struct_name)])

    for name, model in nll_models.items():
        pred = model.predict(batch, verbose=0)[0, :h, :w, :]
        mu = pred[..., 0]
        sigma = sigma_for(pred[..., 1])
        struct_delta = structural_delta_map(ir, mu)

        raw_name = f"{name} [raw delta]"
        struct_name = f"{name} [structural delta]"
        z_name = f"{name} [|z| raw/sigma]"
        struct_z_name = f"{name} [structural z]"

        signals[raw_name] = np.abs(ir - mu)
        signals[struct_name] = struct_delta
        signals[z_name] = np.abs(learned_zscore(ir, mu, sigma))
        signals[struct_z_name] = structural_zscore(struct_delta, sigma)
        STAGES.setdefault(
            name,
            [
                ("raw", raw_name),
                ("structural", struct_name),
                ("|z| = raw/sigma", z_name),
                ("structural z", struct_z_name),
            ],
        )
    return signals

## 4. Score every signal against every ground-truth image


In [6]:
auroc_rows: dict[str, list[float]] = {}

for rgb_path, ir_path in eval_pairs:
    rgb, ir = load_pair(rgb_path, ir_path)
    mask = load_mask(rgb_path.stem)
    signals = signals_for(rgb, ir)

    for name, result in rank_signals(signals, mask).items():
        auroc_rows.setdefault(name, []).append(result.auroc)

    print(f"{rgb_path.stem}: {len(signals)} signals scored")

auroc_mean = {name: float(np.mean(v)) for name, v in auroc_rows.items()}

2026-08-26 12:51:19.675463: I tensorflow/core/grappler/optimizers/custom_graph_optimizer_registry.cc:117] Plugin optimizer for device_type GPU is enabled.


GT01: 30 signals scored
GT02: 30 signals scored
GT03: 30 signals scored


## 5. Per-model evolution

For each model, AUROC at every stage of its own refinement sequence, with the change from the previous stage — a positive `Δ` means that refinement helped *this specific model*, not just the field average. Two columns matter most: `Δ structural` (does isolating SSIM structure help — available to every model) and `Δ sigma-norm` (does dividing by sigma help on top of that — NLL only).


In [7]:
col_w = 24
stage_labels = ["raw", "structural", "|z| = raw/sigma", "structural z"]

header = "model".ljust(col_w) + "".join(s.ljust(20) for s in stage_labels)
header += "Δ structural".ljust(16) + "Δ sigma-norm"
print(header)
print("-" * len(header))

for model_name, stages in STAGES.items():
    values = {label: auroc_mean.get(sig_name) for label, sig_name in stages}
    row = model_name.ljust(col_w)
    for label in stage_labels:
        v = values.get(label)
        row += (f"{v:.4f}" if v is not None else "-").ljust(20)

    delta_structural = (
        values["structural"] - values["raw"]
        if values.get("structural") is not None
        else None
    )
    delta_sigma = (
        values["structural z"] - values["structural"]
        if values.get("structural z") is not None
        else None
    )
    row += (f"{delta_structural:+.4f}" if delta_structural is not None else "-").ljust(
        16
    )
    row += f"{delta_sigma:+.4f}" if delta_sigma is not None else "-"
    print(row)

model                   raw                 structural          |z| = raw/sigma     structural z        Δ structural    Δ sigma-norm
------------------------------------------------------------------------------------------------------------------------------------
unet                    0.4726              0.7073              -                   -                   +0.2347         -
resunet                 0.5520              0.6810              -                   -                   +0.1291         -
attention_unet          0.4993              0.7007              -                   -                   +0.2014         -
efficientnet_unet       0.4810              0.6856              -                   -                   +0.2046         -
efficientnet_unet_ft    0.4614              0.6975              -                   -                   +0.2361         -
unet_nll                0.5860              0.6546              0.6000              0.6712              +0.0687         +0.0

## 6. Overall ranking — every signal, both families together

Ties `040`/`041` together in one list — the answer to "what is the single best reflectography signal in the project right now". `std` across the 3 ground-truth images is the honest error bar; check `040`/`041`'s per-image breakdowns before trusting a narrow win at the top.


In [8]:
auroc_std = {name: float(np.std(v)) for name, v in auroc_rows.items()}
ranking = sorted(auroc_mean, key=lambda n: auroc_mean[n], reverse=True)

col_w2 = 45
print("signal".ljust(col_w2) + "auroc")
print("-" * (col_w2 + 18))
for name in ranking[:20]:
    print(f"{name.ljust(col_w2)}{auroc_mean[name]:.4f} ± {auroc_std[name]:.3f}")

signal                                       auroc
---------------------------------------------------------------
resunet_nll [structural z]                   0.7128 ± 0.092
attention_unet_nll [structural z]            0.7115 ± 0.082
unet [structural delta]                      0.7073 ± 0.077
attention_unet [structural delta]            0.7007 ± 0.042
efficientnet_unet_ft [structural delta]      0.6975 ± 0.059
resunet_nll [structural delta]               0.6914 ± 0.064
attention_unet_nll [structural delta]        0.6906 ± 0.109
efficientnet_unet [structural delta]         0.6856 ± 0.050
efficientnet_unet_nll [structural z]         0.6825 ± 0.081
resunet [structural delta]                   0.6810 ± 0.073
unet_nll [structural z]                      0.6712 ± 0.092
efficientnet_unet_nll [structural delta]     0.6711 ± 0.071
efficientnet_unet_nll_ft [structural z]      0.6635 ± 0.084
unet_nll [structural delta]                  0.6546 ± 0.098
efficientnet_unet_nll_ft [structural delta]  